# R10 — Ticket Classification + SOP Chunks + Vector Search Index

Three artifacts: `hop.genai.ticket_classified` (every support ticket coded by `ai_classify` into the six SOP 01 issue codes), `hop.genai.sop_chunks` (the SOP binder cut into one chunk per `##` section, with chunk ids `<file>.md#<anchor>` and Change Data Feed on), and `hop.genai.sop_chunks_idx` (a Delta Sync Vector Search index over the chunk table, embedding model `databricks-gte-large-en`).

In [0]:
# Config cell — change model names here only, if a built-in model name changes
CATALOG = "hop"
CHAT_MODEL = "databricks-meta-llama-3-3-70b-instruct"
EMBED_MODEL = "databricks-gte-large-en"
AS_OF_DATE = "2026-08-31"
SOP_DIR = f"/Volumes/{CATALOG}/raw/landing/sop_docs"
VS_ENDPOINT = "hungryhop-vs"
print(CATALOG, CHAT_MODEL, EMBED_MODEL, AS_OF_DATE, VS_ENDPOINT)

## Plan

1. **`hop.genai.sop_chunks`** — parse each SOP markdown file from the landing volume; one chunk per `##` section. `chunk_id = <filename>#<anchor>` where the anchor is the `##` heading lowercased with every run of non-alphanumeric characters collapsed to one hyphen (no leading/trailing hyphen). `text` (the embedding source) = the doc's `#` title + `Applies to:` line, then the `##` heading and the section body. `chunk` = the body alone. `delta.enableChangeDataFeed = true` and a `PRIMARY KEY` on `chunk_id` (both required by the Delta Sync index).
2. **`hop.genai.ticket_classified`** — `ai_classify` over each ticket's free-text description; the six SOP 01 issue codes are the labels, each carrying the SOP 01 wording as its description. One row per ticket, `predicted_issue_code` extracted from the v2.1 response.
3. **`hop.genai.sop_chunks_idx`** — one Vector Search endpoint (Free Edition allows one), then a Delta Sync index: primary key `chunk_id`, embedding source column `text`, model `databricks-gte-large-en`, TRIGGERED pipeline synced once the chunks land.

In [0]:
# R10a — hop.genai.sop_chunks: one chunk per ## section of the SOP binder
import glob, os, re, json

def anchor(heading: str) -> str:
    h = heading.lower()
    h = re.sub(r"[^a-z0-9]+", "-", h).strip("-")
    return h

rows = []
for path in sorted(glob.glob(os.path.join(SOP_DIR, "*.md"))):
    fname = os.path.basename(path)
    doc_num = int(fname[:2])                       # doc_name as INT: 1..8 from the file prefix
    text_full = open(path).read()
    lines = text_full.splitlines()
    title_line = next(l for l in lines if l.startswith("# "))
    applies_line = next((l for l in lines if l.lower().startswith("applies to:")), "")
    header_ctx = f"{title_line}\n{applies_line}".strip()

    # split body into ## sections (header preamble before the first ## is ignored)
    section_starts = [i for i, l in enumerate(lines) if l.startswith("## ")]
    for k, start in enumerate(section_starts):
        heading = lines[start][3:].strip()
        end = section_starts[k + 1] if k + 1 < len(section_starts) else len(lines)
        body = "\n".join(lines[start + 1:end]).strip()
        cid = f"{fname}#{anchor(heading)}"
        rows.append({
            "chunk_id": cid,
            "doc_name": doc_num,
            "heading": heading,
            "chunk": body,
            "text": f"{header_ctx}\n\n## {heading}\n\n{body}",
        })

print("chunks parsed:", len(rows))
print("sample ids:", [r["chunk_id"] for r in rows[:3]], "...", rows[-1]["chunk_id"])

schema = "chunk_id STRING, doc_name INT, heading STRING, chunk STRING, text STRING"
chunks_df = spark.createDataFrame(rows, schema)

chunks_df.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{CATALOG}.genai.sop_chunks")

# Delta Sync index prerequisites: CDF + primary key on chunk_id
spark.sql(f"ALTER TABLE {CATALOG}.genai.sop_chunks "
          f"SET TBLPROPERTIES (delta.enableChangeDataFeed = true)")
spark.sql(f"ALTER TABLE {CATALOG}.genai.sop_chunks ALTER COLUMN chunk_id SET NOT NULL")
try:
    spark.sql(f"ALTER TABLE {CATALOG}.genai.sop_chunks "
              f"ADD CONSTRAINT sop_chunks_pk PRIMARY KEY (chunk_id)")
except Exception as e:
    print("PK constraint note:", e)

spark.sql(f"SELECT count(*) AS chunks, count(distinct chunk_id) AS distinct_ids "
          f"FROM {CATALOG}.genai.sop_chunks").show()

In [0]:
# R10b — hop.genai.ticket_classified: ai_classify each ticket into the six SOP 01 issue codes
# (single ai_classify call per ticket, stored once — results are then read from this table)
from pyspark.sql import functions as F
import json

labels_dict = {
    "KITCHEN_DELAY": "the food was not ready when the rider arrived",
    "WRONG_ITEM": "a wrong or missing item in the order",
    "RIDER_DELAY": "the rider was slow or took a wrong route",
    "DAMAGED_IN_TRANSIT": "the food was spilled or crushed on the way",
    "CUSTOMER_UNAVAILABLE": "the customer could not be reached or gave a wrong address",
    "OTHER": "anything else that does not fit the five specific codes",
}

instructions = ("Support tickets are written by HungryHop customers about food delivery orders. "
                "Classify each ticket into the single issue code whose SOP definition best matches "
                "what the customer describes.")

# labels as a JSON object embedded in single-quoted SQL literal (no single quotes inside)
label_sql = "'" + json.dumps(labels_dict) + "'"
classify_expr = "ai_classify(description, " + label_sql \
    + ", MAP('version', '2.1', 'instructions', '" + instructions + "'))"

tickets = spark.table(f"{CATALOG}.raw.support_tickets").select("ticket_id", "description")

classified = (tickets
              .withColumn("classification", F.expr(classify_expr))
              .withColumn("predicted_issue_code", F.expr("classification:response[0].value::STRING"))
              .select("ticket_id", "predicted_issue_code"))

classified.write.mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{CATALOG}.genai.ticket_classified")

tc = spark.table(f"{CATALOG}.genai.ticket_classified")
print("classified rows:", tc.count(), "| null predictions:", tc.filter(F.col("predicted_issue_code").isNull()).count())
tc.groupBy("predicted_issue_code").count().orderBy(F.desc("count")).show()

In [0]:
# R10c — one Vector Search endpoint + Delta Sync index over hop.genai.sop_chunks
from databricks.sdk import WorkspaceClient
from databricks.sdk.service import vectorsearch
import time

w = WorkspaceClient()

existing = [e for e in w.vector_search_endpoints.list_endpoints() if e.name == VS_ENDPOINT]
if existing:
    endpoint = existing[0]
    print("endpoint exists:", endpoint.name, endpoint.endpoint_status.state)
else:
    endpoint = w.vector_search_endpoints.create_endpoint(
        name=VS_ENDPOINT, endpoint_type=vectorsearch.EndpointType.STANDARD)
    print("endpoint creating:", VS_ENDPOINT)

for _ in range(60):
    endpoint = w.vector_search_endpoints.get_endpoint(endpoint_name=VS_ENDPOINT)
    if "ONLINE" in str(endpoint.endpoint_status.state).upper():
        break
    time.sleep(10)
print("endpoint state:", endpoint.endpoint_status.state)

INDEX_NAME = f"{CATALOG}.genai.sop_chunks_idx"
idx = None
try:
    idx = w.vector_search_indexes.get_index(index_name=INDEX_NAME)
    print("index exists:", INDEX_NAME)
except Exception:
    idx = w.vector_search_indexes.create_index(
        name=INDEX_NAME,
        endpoint_name=VS_ENDPOINT,
        primary_key="chunk_id",
        index_type=vectorsearch.VectorIndexType.DELTA_SYNC,
        delta_sync_index_spec=vectorsearch.DeltaSyncVectorIndexSpecRequest(
            source_table=f"{CATALOG}.genai.sop_chunks",
            embedding_source_columns=[
                vectorsearch.EmbeddingSourceColumn(
                    name="text",
                    embedding_model_endpoint_name=EMBED_MODEL),
            ],
            pipeline_type=vectorsearch.PipelineType.TRIGGERED,
        ))
    print("index creating:", INDEX_NAME)

def index_state(i):
    st = getattr(i, "status", None)
    return (getattr(st, "ready", None), getattr(st, "indexed_row_count", None),
            getattr(st, "message", None) or getattr(st, "detailed_state", None) or st)

# wait for the index to provision (sync is rejected before that), then sync and wait for ingestion
for _ in range(90):
    is_ready, rows, msg = index_state(w.vector_search_indexes.get_index(index_name=INDEX_NAME))
    print("index status:", msg, "| rows:", rows)
    if is_ready:
        break
    if msg and ("FAILED" in str(msg).upper() or "CANCELLED" in str(msg).upper()):
        raise RuntimeError(f"index failed to provision: {msg}")
    time.sleep(15)

try:
    w.vector_search_indexes.sync_index(index_name=INDEX_NAME)
except Exception as e:
    print("sync note:", e)

for _ in range(90):
    is_ready, rows, msg = index_state(w.vector_search_indexes.get_index(index_name=INDEX_NAME))
    print("index status:", msg, "| rows:", rows)
    if is_ready and rows is not None and int(rows) >= 36:
        break
    time.sleep(15)
print("index:", INDEX_NAME, "| endpoint:", VS_ENDPOINT)

In [0]:
# R10 verification — chunk contract, coding agreement, index readiness
from pyspark.sql import functions as F
import re

# 1. chunk table: row count, unique ids, CDF property, PK, id format
sc = spark.table(f"{CATALOG}.genai.sop_chunks")
print("chunks:", sc.count())
props = {r[0]: r[1] for r in spark.sql(
    f"SHOW TBLPROPERTIES {CATALOG}.genai.sop_chunks").collect()}
print("CDF enabled:", props.get("delta.enableChangeDataFeed"))

id_ok = sc.filter(F.col("chunk_id").rlike("^[^#]+#[a-z0-9]+(-[a-z0-9]+)*$")).count()
print("chunk_ids matching the anchor grammar:", id_ok, "of", sc.count())

text_ok = sc.filter((F.col("text").startswith("# ")) & F.col("text").contains("Applies to:")
                    & F.col("text").contains("## ")).count()
print("text = title + Applies-to + ## heading + body:", text_ok, "of", sc.count())
print("sample chunk_id:", sc.first()["chunk_id"])

# 2. ticket classification: coverage + agreement with the agent-coded issue_code
tc = spark.table(f"{CATALOG}.genai.ticket_classified")
joined = (spark.table(f"{CATALOG}.raw.support_tickets").select("ticket_id", "issue_code")
          .join(tc, "ticket_id"))
print("classified rows:", tc.count(), "| rows in support_tickets:", 1200)
agree = joined.filter(F.col("predicted_issue_code") == F.col("issue_code")).count()
total = joined.count()
print(f"ai_classify agreement with agent codes: {agree}/{total} = {agree/total:.1%}")
display(joined.groupBy("issue_code", "predicted_issue_code").count()
        .orderBy("issue_code", F.desc("count")))

# 3. index readiness + a smoke retrieval
from databricks.sdk import WorkspaceClient
w = WorkspaceClient()
idx = w.vector_search_indexes.get_index(index_name=f"{CATALOG}.genai.sop_chunks_idx")
print("index status:", getattr(idx, "status", None))
res = w.vector_search_indexes.query_index(
    index_name=f"{CATALOG}.genai.sop_chunks_idx",
    columns=["chunk_id", "heading"],
    query_text="A question about Instant orders and the late delivery compensation",
    num_results=5)
print("top-5 for an Instant-compensation question:")
for d in res.result.data_array:
    print("  ", d)
